In [110]:

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("HospitalAnalytics").getOrCreate()
sc = spark.sparkContext
from pyspark.sql.functions import *
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
doctors = spark.read.csv("doctors.csv", header=True, inferSchema=True)
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
bills = spark.read.csv("bills.csv", header=True, inferSchema=True)
logs = spark.sparkContext.textFile("hospital_logs.txt")
profiles = spark.read.json("patient_profiles.json", multiLine=True)


## Section 1 — DataFrame Basics

In [5]:
# 1
patients.show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [6]:
# 2
doctors.show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      107|   Dr Verma|   Dermatology|     Pune|             850|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+-----------+--------------+---------+----------------+



In [7]:
# 3
appointments.show()

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
|             6|         6|      105|      2024-03-03|Completed|
|             7|         7|      107|      2024-03-04|Cancelled|
|             8|         8|      108|      2024-03-04|Completed|
|             9|         9|      102|      2024-03-05|Completed|
|            10|        10|      103|      2024-03-05|Completed|
|            11|        11|      101|      2024-03-06|  Pending|
|            12|        12|      104|      2024-03-06|Completed|
|            13|         

In [8]:
# 4
bills.show()


+-------+--------------+-----------+------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+-------+--------------+-----------+------------+--------------+
|      1|             1|       1200|         UPI|          Paid|
|      2|             2|        800| Credit Card|          Paid|
|      3|             3|       1500|        Cash|          Paid|
|      4|             4|        900|         UPI|       Pending|
|      5|             5|       1300|  Debit Card|          Paid|
|      6|             6|       2000| Credit Card|          Paid|
|      7|             7|        850|        Cash|     Cancelled|
|      8|             8|       1400|         UPI|          Paid|
|      9|             9|        800|         UPI|          Paid|
|     10|            10|       1500| Credit Card|          Paid|
|     11|            11|       1200|         UPI|       Pending|
|     12|            12|        900|        Cash|          Paid|
|     13|            13| 

In [9]:
# 5
patients.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registration_date: date (nullable = true)



In [10]:
# 6
doctors.printSchema()

root
 |-- doctor_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- consultation_fee: integer (nullable = true)



In [11]:
# 7
appointments.printSchema()

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_id: integer (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- status: string (nullable = true)



In [12]:
# 8
patients.count()

12

In [13]:
# 9
doctors.count()

8

In [14]:
# 10
appointments.show(5)


+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
+--------------+----------+---------+----------------+---------+
only showing top 5 rows


## Section 2 — Select, Rename, Filter

In [15]:
# 11
patients.select("name","city","age").show()


+------+---------+---+
|  name|     city|age|
+------+---------+---+
| Aarav|Hyderabad| 29|
| Priya|Bangalore| 34|
| Rahul|   Mumbai| 41|
| Sneha|    Delhi| 26|
| Kiran|  Chennai| 37|
| Meera|Hyderabad| 31|
|  Amit|     Pune| 45|
|  Neha|    Delhi| 28|
| Divya|Bangalore| 33|
|Vikram|   Mumbai| 52|
|Farhan|Hyderabad| 39|
|Simran|    Delhi| 25|
+------+---------+---+



In [16]:
# 12
doctors.select("doctor_name","specialization","consultation_fee").show()


+-----------+--------------+----------------+
|doctor_name|specialization|consultation_fee|
+-----------+--------------+----------------+
|  Dr Sharma|    Cardiology|            1200|
|    Dr Iyer|   Dermatology|             800|
|    Dr Khan|   Orthopedics|            1500|
|   Dr Reddy|    Pediatrics|             900|
|   Dr Mehta|     Neurology|            2000|
|    Dr Nair|    Cardiology|            1300|
|   Dr Verma|   Dermatology|             850|
|     Dr Rao|   Orthopedics|            1400|
+-----------+--------------+----------------+



In [17]:
# 13
patients.withColumnRenamed("name","patient_name").show()


+----------+------------+---------+---+------+-----------------+
|patient_id|patient_name|     city|age|gender|registration_date|
+----------+------------+---------+---+------+-----------------+
|         1|       Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2|       Priya|Bangalore| 34|Female|       2023-02-12|
|         3|       Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4|       Sneha|    Delhi| 26|Female|       2023-04-15|
|         5|       Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6|       Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|        Amit|     Pune| 45|  Male|       2023-06-22|
|         8|        Neha|    Delhi| 28|Female|       2023-07-10|
|         9|       Divya|Bangalore| 33|Female|       2023-07-15|
|        10|      Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|      Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|      Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------------+

In [18]:
# 14
doctors.withColumnRenamed("doctor_name","consultant_name").show()


+---------+---------------+--------------+---------+----------------+
|doctor_id|consultant_name|specialization|     city|consultation_fee|
+---------+---------------+--------------+---------+----------------+
|      101|      Dr Sharma|    Cardiology|Hyderabad|            1200|
|      102|        Dr Iyer|   Dermatology|Bangalore|             800|
|      103|        Dr Khan|   Orthopedics|   Mumbai|            1500|
|      104|       Dr Reddy|    Pediatrics|    Delhi|             900|
|      105|       Dr Mehta|     Neurology|Hyderabad|            2000|
|      106|        Dr Nair|    Cardiology|  Chennai|            1300|
|      107|       Dr Verma|   Dermatology|     Pune|             850|
|      108|         Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+---------------+--------------+---------+----------------+



In [19]:
# 15
patients.filter(col("city")=="Hyderabad").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
+----------+------+---------+---+------+-----------------+



In [20]:
# 16
patients.filter(col("gender")=="Female").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [21]:
# 17
patients.filter(col("age")>35).show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
+----------+------+---------+---+------+-----------------+



In [22]:
# 18
doctors.filter(col("city")=="Hyderabad").show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
+---------+-----------+--------------+---------+----------------+



In [23]:
# 19
doctors.filter(col("specialization")=="Cardiology").show()


+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
+---------+-----------+--------------+---------+----------------+



In [25]:
# 20
doctors.filter(doctors.consultation_fee > 1000).show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+-----------+--------------+---------+----------------+



## Section 3 — Sorting and Limit

In [26]:
# 21
patients.orderBy("age").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+----------+------+---------+---+------+-----------------+



In [27]:
# 22
patients.orderBy(col("age").desc()).show()


+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [28]:
# 23
patients.orderBy(col("age").desc()).show(5)

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
+----------+------+---------+---+------+-----------------+
only showing top 5 rows


In [29]:
# 24
patients.orderBy("age").show(3)


+----------+------+-----+---+------+-----------------+
|patient_id|  name| city|age|gender|registration_date|
+----------+------+-----+---+------+-----------------+
|        12|Simran|Delhi| 25|Female|       2023-08-21|
|         4| Sneha|Delhi| 26|Female|       2023-04-15|
|         8|  Neha|Delhi| 28|Female|       2023-07-10|
+----------+------+-----+---+------+-----------------+
only showing top 3 rows


In [30]:
# 25
doctors.orderBy(col("consultation_fee").desc()).show()


+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|
|      107|   Dr Verma|   Dermatology|     Pune|             850|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
+---------+-----------+--------------+---------+----------------+



In [31]:
# 26
doctors.orderBy(col("consultation_fee").desc()).show(3)

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+-----------+--------------+---------+----------------+
only showing top 3 rows


In [32]:
# 27
doctors.orderBy("consultation_fee").show()


+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
|      107|   Dr Verma|   Dermatology|     Pune|             850|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
+---------+-----------+--------------+---------+----------------+



In [33]:
# 28
appointments.orderBy("appointment_date").show()


+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
|             6|         6|      105|      2024-03-03|Completed|
|             7|         7|      107|      2024-03-04|Cancelled|
|             8|         8|      108|      2024-03-04|Completed|
|             9|         9|      102|      2024-03-05|Completed|
|            10|        10|      103|      2024-03-05|Completed|
|            11|        11|      101|      2024-03-06|  Pending|
|            12|        12|      104|      2024-03-06|Completed|
|            13|         

In [34]:
# 29
bills.orderBy(col("bill_amount").desc()).show()


+-------+--------------+-----------+------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+-------+--------------+-----------+------------+--------------+
|      6|             6|       2000| Credit Card|          Paid|
|     13|            13|       2000| Credit Card|          Paid|
|      3|             3|       1500|        Cash|          Paid|
|     10|            10|       1500| Credit Card|          Paid|
|      8|             8|       1400|         UPI|          Paid|
|     14|            14|       1400|         UPI|          Paid|
|      5|             5|       1300|  Debit Card|          Paid|
|     16|            16|       1300|  Debit Card|          Paid|
|      1|             1|       1200|         UPI|          Paid|
|     11|            11|       1200|         UPI|       Pending|
|     15|            15|       1200|        Cash|     Cancelled|
|      4|             4|        900|         UPI|       Pending|
|     12|            12| 

In [35]:
# 30
bills.orderBy(col("bill_amount").desc()).show(5)


+-------+--------------+-----------+------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+-------+--------------+-----------+------------+--------------+
|     13|            13|       2000| Credit Card|          Paid|
|      6|             6|       2000| Credit Card|          Paid|
|      3|             3|       1500|        Cash|          Paid|
|     10|            10|       1500| Credit Card|          Paid|
|      8|             8|       1400|         UPI|          Paid|
+-------+--------------+-----------+------------+--------------+
only showing top 5 rows


## Section 4 — Aggregations

In [36]:
# 31
patients.groupBy("city").count().show()


+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [37]:
# 32
patients.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    6|
|  Male|    6|
+------+-----+



In [38]:
# 33
doctors.groupBy("specialization").count().show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|     Neurology|    1|
|   Dermatology|    2|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Orthopedics|    2|
+--------------+-----+



In [42]:
# 34
patients.select(avg("age")).show()

+--------+
|avg(age)|
+--------+
|    35.0|
+--------+



In [40]:
# 35
patients.select(max("age")).show()


+--------+
|max(age)|
+--------+
|      52|
+--------+



In [45]:
# 36
patients.select(min("age")).show()


+--------+
|min(age)|
+--------+
|      25|
+--------+



In [46]:
# 37
doctors.select(avg("consultation_fee")).show()

+---------------------+
|avg(consultation_fee)|
+---------------------+
|              1243.75|
+---------------------+



In [47]:
# 38
doctors.select(max("consultation_fee")).show()


+---------------------+
|max(consultation_fee)|
+---------------------+
|                 2000|
+---------------------+



In [48]:
# 39
bills.select(sum("bill_amount")).show()

+----------------+
|sum(bill_amount)|
+----------------+
|           20250|
+----------------+



In [49]:
# 40
bills.select(avg("bill_amount")).show()

+----------------+
|avg(bill_amount)|
+----------------+
|        1265.625|
+----------------+



## Section 5 — GroupBy Analytics

In [50]:
# 41
patients.groupBy("city").count().show()


+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [51]:
# 42
doctors.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|  Chennai|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    2|
+---------+-----+



In [52]:
# 43
appointments.groupBy("status").count().show()


+---------+-----+
|   status|count|
+---------+-----+
|Completed|   12|
|Cancelled|    2|
|  Pending|    2|
+---------+-----+



In [53]:
# 44
bills.groupBy("payment_status").sum("bill_amount").show()

+--------------+----------------+
|payment_status|sum(bill_amount)|
+--------------+----------------+
|     Cancelled|            2050|
|       Pending|            2100|
|          Paid|           16100|
+--------------+----------------+



In [55]:
# 45
bills.groupBy("payment_mode").sum("bill_amount").show()


+------------+----------------+
|payment_mode|sum(bill_amount)|
+------------+----------------+
| Credit Card|            6300|
|        Cash|            4450|
|  Debit Card|            2600|
|         UPI|            6900|
+------------+----------------+



In [56]:
# 46
bills.groupBy("payment_mode").avg("bill_amount").show()

+------------+----------------+
|payment_mode|avg(bill_amount)|
+------------+----------------+
| Credit Card|          1575.0|
|        Cash|          1112.5|
|  Debit Card|          1300.0|
|         UPI|          1150.0|
+------------+----------------+



In [57]:
# 47
appointments.groupBy("doctor_id").count().show()

+---------+-----+
|doctor_id|count|
+---------+-----+
|      108|    2|
|      101|    3|
|      103|    2|
|      107|    1|
|      102|    2|
|      105|    2|
|      106|    2|
|      104|    2|
+---------+-----+



In [58]:
# 48
appointments.groupBy("patient_id").count().show()

+----------+-----+
|patient_id|count|
+----------+-----+
|        12|    1|
|         1|    2|
|         6|    2|
|         3|    2|
|         5|    1|
|         9|    2|
|         4|    1|
|         8|    1|
|         7|    1|
|        10|    1|
|        11|    1|
|         2|    1|
+----------+-----+



In [59]:
# 49
bills.groupBy("appointment_id").sum("bill_amount").show()


+--------------+----------------+
|appointment_id|sum(bill_amount)|
+--------------+----------------+
|            12|             900|
|             1|            1200|
|            13|            2000|
|             6|            2000|
|            16|            1300|
|             3|            1500|
|             5|            1300|
|            15|            1200|
|             9|             800|
|             4|             900|
|             8|            1400|
|             7|             850|
|            10|            1500|
|            11|            1200|
|            14|            1400|
|             2|             800|
+--------------+----------------+



In [60]:
# 50
doctors.groupBy("specialization").avg("consultation_fee").show()


+--------------+---------------------+
|specialization|avg(consultation_fee)|
+--------------+---------------------+
|     Neurology|               2000.0|
|   Dermatology|                825.0|
|    Cardiology|               1250.0|
|    Pediatrics|                900.0|
|   Orthopedics|               1450.0|
+--------------+---------------------+



## Section 6 — Joins

In [63]:
# 51
patients.join(appointments,"patient_id").show()

+----------+------+---------+---+------+-----------------+--------------+---------+----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|doctor_id|appointment_date|   status|
+----------+------+---------+---+------+-----------------+--------------+---------+----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|      101|      2024-03-01|Completed|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|      102|      2024-03-01|Completed|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|      103|      2024-03-02|Completed|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|             4|      104|      2024-03-02|  Pending|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|             5|      106|      2024-03-03|Completed|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|             6|      105|      2024-03-03|Completed|
|

In [64]:
# 52
patients.join(appointments,"patient_id").select("name","city","appointment_date","status").show()


+------+---------+----------------+---------+
|  name|     city|appointment_date|   status|
+------+---------+----------------+---------+
| Aarav|Hyderabad|      2024-03-01|Completed|
| Priya|Bangalore|      2024-03-01|Completed|
| Rahul|   Mumbai|      2024-03-02|Completed|
| Sneha|    Delhi|      2024-03-02|  Pending|
| Kiran|  Chennai|      2024-03-03|Completed|
| Meera|Hyderabad|      2024-03-03|Completed|
|  Amit|     Pune|      2024-03-04|Cancelled|
|  Neha|    Delhi|      2024-03-04|Completed|
| Divya|Bangalore|      2024-03-05|Completed|
|Vikram|   Mumbai|      2024-03-05|Completed|
|Farhan|Hyderabad|      2024-03-06|  Pending|
|Simran|    Delhi|      2024-03-06|Completed|
| Aarav|Hyderabad|      2024-03-07|Completed|
| Rahul|   Mumbai|      2024-03-07|Completed|
| Meera|Hyderabad|      2024-03-08|Cancelled|
| Divya|Bangalore|      2024-03-08|Completed|
+------+---------+----------------+---------+



In [65]:
# 53
doctors.join(appointments,"doctor_id").show()

+---------+-----------+--------------+---------+----------------+--------------+----------+----------------+---------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|appointment_id|patient_id|appointment_date|   status|
+---------+-----------+--------------+---------+----------------+--------------+----------+----------------+---------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|             1|         1|      2024-03-01|Completed|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|             2|         2|      2024-03-01|Completed|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|             3|         3|      2024-03-02|Completed|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|             4|         4|      2024-03-02|  Pending|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|             5|         5|      2024-03-03|Completed|
|      105|   Dr Mehta|     Neurology|Hyderabad|

In [66]:
# 54
doctors.join(appointments,"doctor_id").select("doctor_name","specialization","appointment_date","status").show()


+-----------+--------------+----------------+---------+
|doctor_name|specialization|appointment_date|   status|
+-----------+--------------+----------------+---------+
|  Dr Sharma|    Cardiology|      2024-03-01|Completed|
|    Dr Iyer|   Dermatology|      2024-03-01|Completed|
|    Dr Khan|   Orthopedics|      2024-03-02|Completed|
|   Dr Reddy|    Pediatrics|      2024-03-02|  Pending|
|    Dr Nair|    Cardiology|      2024-03-03|Completed|
|   Dr Mehta|     Neurology|      2024-03-03|Completed|
|   Dr Verma|   Dermatology|      2024-03-04|Cancelled|
|     Dr Rao|   Orthopedics|      2024-03-04|Completed|
|    Dr Iyer|   Dermatology|      2024-03-05|Completed|
|    Dr Khan|   Orthopedics|      2024-03-05|Completed|
|  Dr Sharma|    Cardiology|      2024-03-06|  Pending|
|   Dr Reddy|    Pediatrics|      2024-03-06|Completed|
|   Dr Mehta|     Neurology|      2024-03-07|Completed|
|     Dr Rao|   Orthopedics|      2024-03-07|Completed|
|  Dr Sharma|    Cardiology|      2024-03-08|Can

In [67]:
# 55
appointments.join(bills,"appointment_id").show()

+--------------+----------+---------+----------------+---------+-------+-----------+------------+--------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|bill_id|bill_amount|payment_mode|payment_status|
+--------------+----------+---------+----------------+---------+-------+-----------+------------+--------------+
|             1|         1|      101|      2024-03-01|Completed|      1|       1200|         UPI|          Paid|
|             2|         2|      102|      2024-03-01|Completed|      2|        800| Credit Card|          Paid|
|             3|         3|      103|      2024-03-02|Completed|      3|       1500|        Cash|          Paid|
|             4|         4|      104|      2024-03-02|  Pending|      4|        900|         UPI|       Pending|
|             5|         5|      106|      2024-03-03|Completed|      5|       1300|  Debit Card|          Paid|
|             6|         6|      105|      2024-03-03|Completed|      6|       2000| Credit Card

In [68]:
# 56
appointments.join(bills,"appointment_id").select("appointment_id","status","bill_amount","payment_status").show()

+--------------+---------+-----------+--------------+
|appointment_id|   status|bill_amount|payment_status|
+--------------+---------+-----------+--------------+
|             1|Completed|       1200|          Paid|
|             2|Completed|        800|          Paid|
|             3|Completed|       1500|          Paid|
|             4|  Pending|        900|       Pending|
|             5|Completed|       1300|          Paid|
|             6|Completed|       2000|          Paid|
|             7|Cancelled|        850|     Cancelled|
|             8|Completed|       1400|          Paid|
|             9|Completed|        800|          Paid|
|            10|Completed|       1500|          Paid|
|            11|  Pending|       1200|       Pending|
|            12|Completed|        900|          Paid|
|            13|Completed|       2000|          Paid|
|            14|Completed|       1400|          Paid|
|            15|Cancelled|       1200|     Cancelled|
|            16|Completed|  

In [69]:
# 57
patients.join(appointments,"patient_id").join(doctors,"doctor_id").show()

+---------+----------+------+---------+---+------+-----------------+--------------+----------------+---------+-----------+--------------+---------+----------------+
|doctor_id|patient_id|  name|     city|age|gender|registration_date|appointment_id|appointment_date|   status|doctor_name|specialization|     city|consultation_fee|
+---------+----------+------+---------+---+------+-----------------+--------------+----------------+---------+-----------+--------------+---------+----------------+
|      101|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|      2024-03-01|Completed|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      102|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|      2024-03-01|Completed|    Dr Iyer|   Dermatology|Bangalore|             800|
|      103|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|      2024-03-02|Completed|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      104

In [70]:
# 58
patients.join(appointments,"patient_id").join(doctors,"doctor_id") \
.select("name","doctor_name","specialization","appointment_date").show()


+------+-----------+--------------+----------------+
|  name|doctor_name|specialization|appointment_date|
+------+-----------+--------------+----------------+
| Aarav|  Dr Sharma|    Cardiology|      2024-03-01|
| Priya|    Dr Iyer|   Dermatology|      2024-03-01|
| Rahul|    Dr Khan|   Orthopedics|      2024-03-02|
| Sneha|   Dr Reddy|    Pediatrics|      2024-03-02|
| Kiran|    Dr Nair|    Cardiology|      2024-03-03|
| Meera|   Dr Mehta|     Neurology|      2024-03-03|
|  Amit|   Dr Verma|   Dermatology|      2024-03-04|
|  Neha|     Dr Rao|   Orthopedics|      2024-03-04|
| Divya|    Dr Iyer|   Dermatology|      2024-03-05|
|Vikram|    Dr Khan|   Orthopedics|      2024-03-05|
|Farhan|  Dr Sharma|    Cardiology|      2024-03-06|
|Simran|   Dr Reddy|    Pediatrics|      2024-03-06|
| Aarav|   Dr Mehta|     Neurology|      2024-03-07|
| Rahul|     Dr Rao|   Orthopedics|      2024-03-07|
| Meera|  Dr Sharma|    Cardiology|      2024-03-08|
| Divya|    Dr Nair|    Cardiology|      2024-

In [71]:
# 59
patients.join(appointments,"patient_id").join(doctors,"doctor_id").join(bills,"appointment_id").show()


+--------------+---------+----------+------+---------+---+------+-----------------+----------------+---------+-----------+--------------+---------+----------------+-------+-----------+------------+--------------+
|appointment_id|doctor_id|patient_id|  name|     city|age|gender|registration_date|appointment_date|   status|doctor_name|specialization|     city|consultation_fee|bill_id|bill_amount|payment_mode|payment_status|
+--------------+---------+----------+------+---------+---+------+-----------------+----------------+---------+-----------+--------------+---------+----------------+-------+-----------+------------+--------------+
|             1|      101|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|      2024-03-01|Completed|  Dr Sharma|    Cardiology|Hyderabad|            1200|      1|       1200|         UPI|          Paid|
|             2|      102|         2| Priya|Bangalore| 34|Female|       2023-02-12|      2024-03-01|Completed|    Dr Iyer|   Dermatology|Bangalore| 

In [72]:
# 60
patients.join(appointments,"patient_id").join(doctors,"doctor_id").join(bills,"appointment_id") \
.select("name","doctor_name","status","bill_amount","payment_mode").show()

+------+-----------+---------+-----------+------------+
|  name|doctor_name|   status|bill_amount|payment_mode|
+------+-----------+---------+-----------+------------+
| Aarav|  Dr Sharma|Completed|       1200|         UPI|
| Priya|    Dr Iyer|Completed|        800| Credit Card|
| Rahul|    Dr Khan|Completed|       1500|        Cash|
| Sneha|   Dr Reddy|  Pending|        900|         UPI|
| Kiran|    Dr Nair|Completed|       1300|  Debit Card|
| Meera|   Dr Mehta|Completed|       2000| Credit Card|
|  Amit|   Dr Verma|Cancelled|        850|        Cash|
|  Neha|     Dr Rao|Completed|       1400|         UPI|
| Divya|    Dr Iyer|Completed|        800|         UPI|
|Vikram|    Dr Khan|Completed|       1500| Credit Card|
|Farhan|  Dr Sharma|  Pending|       1200|         UPI|
|Simran|   Dr Reddy|Completed|        900|        Cash|
| Aarav|   Dr Mehta|Completed|       2000| Credit Card|
| Rahul|     Dr Rao|Completed|       1400|         UPI|
| Meera|  Dr Sharma|Cancelled|       1200|      

## Section 7 — Updating Data with withColumn

In [73]:
# 61
patients.withColumn("age_group",
    when(col("age")<30,"Young")
    .when(col("age")<40,"Adult")
    .otherwise("Senior")
).show()

+----------+------+---------+---+------+-----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|age_group|
+----------+------+---------+---+------+-----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   Senior|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   Senior|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    Adult|
|        12|Simran|    Delhi| 25|F

In [74]:
# 62
patients.withColumn("hospital_name",lit("BotCampus Hospital")).show()


+----------+------+---------+---+------+-----------------+------------------+
|patient_id|  name|     city|age|gender|registration_date|     hospital_name|
+----------+------+---------+---+------+-----------------+------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|BotCampus Hospital|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|BotCampus Hospital|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|BotCampus Hospital|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|BotCampus Hospital|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|BotCampus Hospital|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|BotCampus Hospital|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|BotCampus Hospital|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|BotCampus Hospital|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|BotCampus Hospital|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|BotCam

In [75]:
# 63
doctors.withColumn("fee_with_tax",col("consultation_fee")*1.18).show()


+---------+-----------+--------------+---------+----------------+------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|fee_with_tax|
+---------+-----------+--------------+---------+----------------+------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|      1416.0|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|       944.0|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|      1770.0|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|      1062.0|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|      2360.0|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|      1534.0|
|      107|   Dr Verma|   Dermatology|     Pune|             850|      1003.0|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|      1652.0|
+---------+-----------+--------------+---------+----------------+------------+



In [76]:
# 64
bills.withColumn("bill_with_tax",col("bill_amount")*1.18).show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_with_tax|
+-------+--------------+-----------+------------+--------------+-------------+
|      1|             1|       1200|         UPI|          Paid|       1416.0|
|      2|             2|        800| Credit Card|          Paid|        944.0|
|      3|             3|       1500|        Cash|          Paid|       1770.0|
|      4|             4|        900|         UPI|       Pending|       1062.0|
|      5|             5|       1300|  Debit Card|          Paid|       1534.0|
|      6|             6|       2000| Credit Card|          Paid|       2360.0|
|      7|             7|        850|        Cash|     Cancelled|       1003.0|
|      8|             8|       1400|         UPI|          Paid|       1652.0|
|      9|             9|        800|         UPI|          Paid|        944.0|
|     10|            10|       1500| Credit Card|   

In [77]:
# 65
bills.withColumn("bill_in_thousands",col("bill_amount")/1000).show()


+-------+--------------+-----------+------------+--------------+-----------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_in_thousands|
+-------+--------------+-----------+------------+--------------+-----------------+
|      1|             1|       1200|         UPI|          Paid|              1.2|
|      2|             2|        800| Credit Card|          Paid|              0.8|
|      3|             3|       1500|        Cash|          Paid|              1.5|
|      4|             4|        900|         UPI|       Pending|              0.9|
|      5|             5|       1300|  Debit Card|          Paid|              1.3|
|      6|             6|       2000| Credit Card|          Paid|              2.0|
|      7|             7|        850|        Cash|     Cancelled|             0.85|
|      8|             8|       1400|         UPI|          Paid|              1.4|
|      9|             9|        800|         UPI|          Paid|              0.8|
|   

In [78]:
# 66
patients.withColumn("country",lit("India")).show()


+----------+------+---------+---+------+-----------------+-------+
|patient_id|  name|     city|age|gender|registration_date|country|
+----------+------+---------+---+------+-----------------+-------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|  India|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|  India|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|  India|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|  India|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|  India|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|  India|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|  India|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|  India|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|  India|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|  India|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|  India|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|  In

In [79]:
# 67
doctors.withColumn("doctor_label",concat(col("doctor_name"),lit(" - "),col("specialization"))).show()

+---------+-----------+--------------+---------+----------------+--------------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|        doctor_label|
+---------+-----------+--------------+---------+----------------+--------------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|Dr Sharma - Cardi...|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|Dr Iyer - Dermato...|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|Dr Khan - Orthope...|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|Dr Reddy - Pediat...|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|Dr Mehta - Neurology|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|Dr Nair - Cardiology|
|      107|   Dr Verma|   Dermatology|     Pune|             850|Dr Verma - Dermat...|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|Dr Rao - Orthopedics|
+---------+-----------+--------------+-----

In [80]:
# 68
patients.withColumn("patient_label",concat(col("name"),lit(" - "),col("city"))).show()


+----------+------+---------+---+------+-----------------+------------------+
|patient_id|  name|     city|age|gender|registration_date|     patient_label|
+----------+------+---------+---+------+-----------------+------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10| Aarav - Hyderabad|
|         2| Priya|Bangalore| 34|Female|       2023-02-12| Priya - Bangalore|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    Rahul - Mumbai|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|     Sneha - Delhi|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   Kiran - Chennai|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10| Meera - Hyderabad|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|       Amit - Pune|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|      Neha - Delhi|
|         9| Divya|Bangalore| 33|Female|       2023-07-15| Divya - Bangalore|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Vik

In [81]:
# 69
bills.withColumn("high_bill_flag",when(col("bill_amount")>1500,1).otherwise(0)).show()


+-------+--------------+-----------+------------+--------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+-------+--------------+-----------+------------+--------------+--------------+
|      1|             1|       1200|         UPI|          Paid|             0|
|      2|             2|        800| Credit Card|          Paid|             0|
|      3|             3|       1500|        Cash|          Paid|             0|
|      4|             4|        900|         UPI|       Pending|             0|
|      5|             5|       1300|  Debit Card|          Paid|             0|
|      6|             6|       2000| Credit Card|          Paid|             1|
|      7|             7|        850|        Cash|     Cancelled|             0|
|      8|             8|       1400|         UPI|          Paid|             0|
|      9|             9|        800|         UPI|          Paid|             0|
|     10|            10|       1500| Cre

In [82]:
# 70
patients.withColumn("senior_patient_flag",when(col("age")>40,1).otherwise(0)).show()

+----------+------+---------+---+------+-----------------+-------------------+
|patient_id|  name|     city|age|gender|registration_date|senior_patient_flag|
+----------+------+---------+---+------+-----------------+-------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|                  0|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|                  0|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|                  1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|                  0|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|                  0|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|                  0|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|                  1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|                  0|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|                  0|
|        10|Vikram|   Mumbai| 52|  Male|       2023-

## Section 8 — When/Otherwise

In [83]:
# 71
patients.withColumn("age_category",
    when(col("age")<30,"Young")
    .when(col("age")<=40,"Adult")
    .otherwise("Senior")
).show()

+----------+------+---------+---+------+-----------------+------------+
|patient_id|  name|     city|age|gender|registration_date|age_category|
+----------+------+---------+---+------+-----------------+------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|       Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|       Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|      Senior|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|       Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|       Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|       Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|      Senior|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|       Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|       Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|      Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|      

In [84]:
# 72
bills.withColumn("bill_category",
    when(col("bill_amount")>1500,"High")
    .when(col("bill_amount")>=1000,"Medium")
    .otherwise("Low")
).show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_category|
+-------+--------------+-----------+------------+--------------+-------------+
|      1|             1|       1200|         UPI|          Paid|       Medium|
|      2|             2|        800| Credit Card|          Paid|          Low|
|      3|             3|       1500|        Cash|          Paid|       Medium|
|      4|             4|        900|         UPI|       Pending|          Low|
|      5|             5|       1300|  Debit Card|          Paid|       Medium|
|      6|             6|       2000| Credit Card|          Paid|         High|
|      7|             7|        850|        Cash|     Cancelled|          Low|
|      8|             8|       1400|         UPI|          Paid|       Medium|
|      9|             9|        800|         UPI|          Paid|          Low|
|     10|            10|       1500| Credit Card|   

In [85]:
# 73
doctors.withColumn("fee_category",
    when(col("consultation_fee")>1500,"Premium")
    .when(col("consultation_fee")>=1000,"Standard")
    .otherwise("Basic")
).show()

+---------+-----------+--------------+---------+----------------+------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|fee_category|
+---------+-----------+--------------+---------+----------------+------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|    Standard|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|       Basic|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|    Standard|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|       Basic|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|     Premium|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|    Standard|
|      107|   Dr Verma|   Dermatology|     Pune|             850|       Basic|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|    Standard|
+---------+-----------+--------------+---------+----------------+------------+



In [86]:
# 74
appointments.withColumn("priority",
    when(col("status")=="Pending","High")
    .when(col("status")=="Completed","Normal")
    .otherwise("Low")
).show()

+--------------+----------+---------+----------------+---------+--------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|priority|
+--------------+----------+---------+----------------+---------+--------+
|             1|         1|      101|      2024-03-01|Completed|  Normal|
|             2|         2|      102|      2024-03-01|Completed|  Normal|
|             3|         3|      103|      2024-03-02|Completed|  Normal|
|             4|         4|      104|      2024-03-02|  Pending|    High|
|             5|         5|      106|      2024-03-03|Completed|  Normal|
|             6|         6|      105|      2024-03-03|Completed|  Normal|
|             7|         7|      107|      2024-03-04|Cancelled|     Low|
|             8|         8|      108|      2024-03-04|Completed|  Normal|
|             9|         9|      102|      2024-03-05|Completed|  Normal|
|            10|        10|      103|      2024-03-05|Completed|  Normal|
|            11|        11|      101| 

In [87]:
# 75
patients.withColumn("zone",
    when(col("city").isin("Hyderabad","Bangalore","Chennai"),"South")
    .otherwise("Metro")
).show()

+----------+------+---------+---+------+-----------------+-----+
|patient_id|  name|     city|age|gender|registration_date| zone|
+----------+------+---------+---+------+-----------------+-----+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|South|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|South|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|Metro|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|Metro|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|South|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|South|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|Metro|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|Metro|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|South|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|Metro|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|South|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|Metro|
+----------+------+------

## Section 9 — Date Functions

In [88]:
# 76
patients.withColumn("registration_date",to_date("registration_date")).show()


+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [89]:
# 77
patients.withColumn("year",year("registration_date")).show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|year|
+----------+------+---------+---+------+-----------------+----+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|2023|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|2023|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|2023|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|2023|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|2023|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|2023|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|2023|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|2023|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|2023|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|2023|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|2023|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|2023|
+----------+------+---------+---+------+

In [90]:
# 78
patients.withColumn("month",month("registration_date")).show()

+----------+------+---------+---+------+-----------------+-----+
|patient_id|  name|     city|age|gender|registration_date|month|
+----------+------+---------+---+------+-----------------+-----+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    1|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    2|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    3|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    4|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    5|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    6|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|    6|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    7|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    7|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|    8|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    8|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|    8|
+----------+------+------

In [91]:
# 79
appointments.withColumn("appointment_date",to_date("appointment_date")).show()

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
|             6|         6|      105|      2024-03-03|Completed|
|             7|         7|      107|      2024-03-04|Cancelled|
|             8|         8|      108|      2024-03-04|Completed|
|             9|         9|      102|      2024-03-05|Completed|
|            10|        10|      103|      2024-03-05|Completed|
|            11|        11|      101|      2024-03-06|  Pending|
|            12|        12|      104|      2024-03-06|Completed|
|            13|         

In [92]:
# 80
appointments.withColumn("month",month("appointment_date")).show()

+--------------+----------+---------+----------------+---------+-----+
|appointment_id|patient_id|doctor_id|appointment_date|   status|month|
+--------------+----------+---------+----------------+---------+-----+
|             1|         1|      101|      2024-03-01|Completed|    3|
|             2|         2|      102|      2024-03-01|Completed|    3|
|             3|         3|      103|      2024-03-02|Completed|    3|
|             4|         4|      104|      2024-03-02|  Pending|    3|
|             5|         5|      106|      2024-03-03|Completed|    3|
|             6|         6|      105|      2024-03-03|Completed|    3|
|             7|         7|      107|      2024-03-04|Cancelled|    3|
|             8|         8|      108|      2024-03-04|Completed|    3|
|             9|         9|      102|      2024-03-05|Completed|    3|
|            10|        10|      103|      2024-03-05|Completed|    3|
|            11|        11|      101|      2024-03-06|  Pending|    3|
|     

In [93]:
# 81
appointments.groupBy("appointment_date").count().show()


+----------------+-----+
|appointment_date|count|
+----------------+-----+
|      2024-03-05|    2|
|      2024-03-06|    2|
|      2024-03-08|    2|
|      2024-03-04|    2|
|      2024-03-02|    2|
|      2024-03-01|    2|
|      2024-03-03|    2|
|      2024-03-07|    2|
+----------------+-----+



In [94]:
# 82
appointments.withColumn("month",month("appointment_date")).groupBy("month").count().show()


+-----+-----+
|month|count|
+-----+-----+
|    3|   16|
+-----+-----+



In [95]:
# 83
patients.filter(col("registration_date")>"2023-06-01").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [96]:
# 84
patients.withColumn("days_since_registration",datediff(current_date(),col("registration_date"))).show()


+----------+------+---------+---+------+-----------------+-----------------------+
|patient_id|  name|     city|age|gender|registration_date|days_since_registration|
+----------+------+---------+---+------+-----------------+-----------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|                   1203|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|                   1170|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|                   1140|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|                   1108|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|                   1082|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|                   1052|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|                   1040|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|                   1022|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|                   1017|
|   

In [97]:
# 85
appointments.withColumn("days_since_appointment",datediff(current_date(),col("appointment_date"))).show()


+--------------+----------+---------+----------------+---------+----------------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|days_since_appointment|
+--------------+----------+---------+----------------+---------+----------------------+
|             1|         1|      101|      2024-03-01|Completed|                   787|
|             2|         2|      102|      2024-03-01|Completed|                   787|
|             3|         3|      103|      2024-03-02|Completed|                   786|
|             4|         4|      104|      2024-03-02|  Pending|                   786|
|             5|         5|      106|      2024-03-03|Completed|                   785|
|             6|         6|      105|      2024-03-03|Completed|                   785|
|             7|         7|      107|      2024-03-04|Cancelled|                   784|
|             8|         8|      108|      2024-03-04|Completed|                   784|
|             9|         9|     

## Section 10 — Window Functions

In [98]:
# 86
from pyspark.sql.window import Window
patients.withColumn("rank",rank().over(Window.partitionBy("city").orderBy(col("age").desc()))).show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|rank|
+----------+------+---------+---+------+-----------------+----+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|   1|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|   2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|   1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|   2|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|   3|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|   1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|   2|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|   3|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   1|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   2|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   1|
+----------+------+---------+---+------+

In [99]:
# 87
patients.withColumn("rn",row_number().over(Window.partitionBy("city").orderBy(col("age").desc()))) \
.filter(col("rn")==1).show()


+----------+------+---------+---+------+-----------------+---+
|patient_id|  name|     city|age|gender|registration_date| rn|
+----------+------+---------+---+------+-----------------+---+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|  1|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|  1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|  1|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|  1|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|  1|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|  1|
+----------+------+---------+---+------+-----------------+---+



In [100]:
# 88
doctors.withColumn("drank",dense_rank().over(Window.partitionBy("specialization").orderBy(col("consultation_fee").desc()))).show()

+---------+-----------+--------------+---------+----------------+-----+
|doctor_id|doctor_name|specialization|     city|consultation_fee|drank|
+---------+-----------+--------------+---------+----------------+-----+
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|    1|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|    2|
|      107|   Dr Verma|   Dermatology|     Pune|             850|    1|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|    2|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|    1|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|    1|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|    2|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|    1|
+---------+-----------+--------------+---------+----------------+-----+



In [101]:
# 89
doctors.withColumn("drank",dense_rank().over(Window.partitionBy("specialization").orderBy(col("consultation_fee").desc()))) \
.filter(col("drank")==1).show()


+---------+-----------+--------------+---------+----------------+-----+
|doctor_id|doctor_name|specialization|     city|consultation_fee|drank|
+---------+-----------+--------------+---------+----------------+-----+
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|    1|
|      107|   Dr Verma|   Dermatology|     Pune|             850|    1|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|    1|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|    1|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|    1|
+---------+-----------+--------------+---------+----------------+-----+



In [102]:
# 90
patients.withColumn("rnk",rank().over(Window.partitionBy("city").orderBy(col("age").desc()))).filter(col("rnk")<=2).show()

+----------+------+---------+---+------+-----------------+---+
|patient_id|  name|     city|age|gender|registration_date|rnk|
+----------+------+---------+---+------+-----------------+---+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|  1|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|  2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|  1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|  1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|  2|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|  1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|  2|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|  1|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|  2|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|  1|
+----------+------+---------+---+------+-----------------+---+



In [103]:
# 91
doctors.withColumn("rank",rank().over(Window.orderBy(col("consultation_fee").desc()))).show()

+---------+-----------+--------------+---------+----------------+----+
|doctor_id|doctor_name|specialization|     city|consultation_fee|rank|
+---------+-----------+--------------+---------+----------------+----+
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|   1|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|   2|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|   3|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|   4|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|   5|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|   6|
|      107|   Dr Verma|   Dermatology|     Pune|             850|   7|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|   8|
+---------+-----------+--------------+---------+----------------+----+



In [104]:
# 92
patients.groupBy("city").count() \
.withColumn("rank",rank().over(Window.orderBy(col("count").desc()))).show()

+---------+-----+----+
|     city|count|rank|
+---------+-----+----+
|    Delhi|    3|   1|
|Hyderabad|    3|   1|
|Bangalore|    2|   3|
|   Mumbai|    2|   3|
|  Chennai|    1|   5|
|     Pune|    1|   5|
+---------+-----+----+



In [105]:
# 93
appointments.groupBy("doctor_id").count().withColumn("rank",rank().over(Window.orderBy(col("count").desc()))).show()

+---------+-----+----+
|doctor_id|count|rank|
+---------+-----+----+
|      101|    3|   1|
|      108|    2|   2|
|      103|    2|   2|
|      102|    2|   2|
|      105|    2|   2|
|      106|    2|   2|
|      104|    2|   2|
|      107|    1|   8|
+---------+-----+----+



In [106]:
# 94
bills.withColumn("running_total",
    sum("bill_amount").over(Window.partitionBy("payment_mode").orderBy("bill_id"))
).show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|running_total|
+-------+--------------+-----------+------------+--------------+-------------+
|      3|             3|       1500|        Cash|          Paid|         1500|
|      7|             7|        850|        Cash|     Cancelled|         2350|
|     12|            12|        900|        Cash|          Paid|         3250|
|     15|            15|       1200|        Cash|     Cancelled|         4450|
|      2|             2|        800| Credit Card|          Paid|          800|
|      6|             6|       2000| Credit Card|          Paid|         2800|
|     10|            10|       1500| Credit Card|          Paid|         4300|
|     13|            13|       2000| Credit Card|          Paid|         6300|
|      5|             5|       1300|  Debit Card|          Paid|         1300|
|     16|            16|       1300|  Debit Card|   

In [107]:
# 95
appointments.withColumn("running_count",
    count("appointment_id").over(Window.partitionBy("doctor_id").orderBy("appointment_id"))
).show()


+--------------+----------+---------+----------------+---------+-------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|running_count|
+--------------+----------+---------+----------------+---------+-------------+
|             1|         1|      101|      2024-03-01|Completed|            1|
|            11|        11|      101|      2024-03-06|  Pending|            2|
|            15|         6|      101|      2024-03-08|Cancelled|            3|
|             2|         2|      102|      2024-03-01|Completed|            1|
|             9|         9|      102|      2024-03-05|Completed|            2|
|             3|         3|      103|      2024-03-02|Completed|            1|
|            10|        10|      103|      2024-03-05|Completed|            2|
|             4|         4|      104|      2024-03-02|  Pending|            1|
|            12|        12|      104|      2024-03-06|Completed|            2|
|             6|         6|      105|      2024-03-0

## Section 11 — RDD Exercises

In [111]:
# 96
logs = sc.textFile("hospital_logs.txt")
logs.count()


20

In [112]:
# 97
logs.map(lambda x: x.split(" ")[0]).collect()

['Aarav',
 'Priya',
 'Rahul',
 'Sneha',
 'Aarav',
 'Kiran',
 'Meera',
 'Vikram',
 'Divya',
 'Farhan',
 'Simran',
 'Neha',
 'Amit',
 'Rahul',
 'Meera',
 'Aarav',
 'Priya',
 'Divya',
 'Vikram',
 'Farhan']

In [113]:
# 98
logs.map(lambda x: x.split(" ")[1]).collect()

['login',
 'login',
 'appointment',
 'login',
 'payment',
 'appointment',
 'login',
 'appointment',
 'payment',
 'login',
 'appointment',
 'payment',
 'login',
 'payment',
 'appointment',
 'logout',
 'payment',
 'login',
 'payment',
 'appointment']

In [114]:
# 99
logs.map(lambda x: x.split(" ")[0]).distinct().collect()

['Kiran',
 'Vikram',
 'Divya',
 'Simran',
 'Amit',
 'Aarav',
 'Priya',
 'Rahul',
 'Sneha',
 'Meera',
 'Farhan',
 'Neha']

In [115]:
# 100
logs.map(lambda x: (x.split(" ")[1],1)).reduceByKey(lambda a,b:a+b).collect()


[('login', 7), ('payment', 6), ('appointment', 6), ('logout', 1)]

## Bonus — Complex JSON Exercises

In [118]:
# 1
profiles.show()


+---------------+--------------------+------+----------+
|      allergies|             contact|  name|patient_id|
+---------------+--------------------+------+----------+
|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav|         1|
|       [Pollen]|{priya@mail.com, ...| Priya|         2|
|   [Dust, Milk]|{rahul@mail.com, ...| Rahul|         3|
|      [Seafood]|{meera@mail.com, ...| Meera|         6|
| [Pollen, Dust]|{vikram@mail.com,...|Vikram|        10|
+---------------+--------------------+------+----------+



In [119]:
# 2
profiles.printSchema()


root
 |-- allergies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- name: string (nullable = true)
 |-- patient_id: long (nullable = true)



In [120]:
# 3
profiles.select("patient_id","name","contact.email","contact.phone").show()


+----------+------+---------------+----------+
|patient_id|  name|          email|     phone|
+----------+------+---------------+----------+
|         1| Aarav| aarav@mail.com|9000011111|
|         2| Priya| priya@mail.com|9000022222|
|         3| Rahul| rahul@mail.com|9000033333|
|         6| Meera| meera@mail.com|9000066666|
|        10|Vikram|vikram@mail.com|9000101010|
+----------+------+---------------+----------+



In [121]:
# 4
profiles.select("patient_id","name",explode("allergies")).show()


+----------+------+-------+
|patient_id|  name|    col|
+----------+------+-------+
|         1| Aarav|   Dust|
|         1| Aarav|Peanuts|
|         2| Priya| Pollen|
|         3| Rahul|   Dust|
|         3| Rahul|   Milk|
|         6| Meera|Seafood|
|        10|Vikram| Pollen|
|        10|Vikram|   Dust|
+----------+------+-------+



In [122]:
# 5
profiles.select(explode("allergies").alias("allergy")).groupBy("allergy").count().show()


+-------+-----+
|allergy|count|
+-------+-----+
|   Milk|    1|
|   Dust|    3|
|Peanuts|    1|
| Pollen|    2|
|Seafood|    1|
+-------+-----+



In [123]:
# 6
profiles.filter(array_contains("allergies","Dust")).show()


+---------------+--------------------+------+----------+
|      allergies|             contact|  name|patient_id|
+---------------+--------------------+------+----------+
|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav|         1|
|   [Dust, Milk]|{rahul@mail.com, ...| Rahul|         3|
| [Pollen, Dust]|{vikram@mail.com,...|Vikram|        10|
+---------------+--------------------+------+----------+



In [124]:
# 7
profiles.select(explode("allergies").alias("allergy")).distinct().show()


+-------+
|allergy|
+-------+
|   Milk|
|   Dust|
|Peanuts|
| Pollen|
|Seafood|
+-------+



In [125]:
# 8
profiles.join(patients,"patient_id").show()


+----------+---------------+--------------------+------+------+---------+---+------+-----------------+
|patient_id|      allergies|             contact|  name|  name|     city|age|gender|registration_date|
+----------+---------------+--------------------+------+------+---------+---+------+-----------------+
|         1|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2|       [Pollen]|{priya@mail.com, ...| Priya| Priya|Bangalore| 34|Female|       2023-02-12|
|         3|   [Dust, Milk]|{rahul@mail.com, ...| Rahul| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         6|      [Seafood]|{meera@mail.com, ...| Meera| Meera|Hyderabad| 31|Female|       2023-06-10|
|        10| [Pollen, Dust]|{vikram@mail.com,...|Vikram|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+----------+---------------+--------------------+------+------+---------+---+------+-----------------+



In [127]:
# 9
profiles.join(patients,"patient_id").select(patients.name.alias("patient_name"),"city","contact.email","allergies").show()

+------------+---------+---------------+---------------+
|patient_name|     city|          email|      allergies|
+------------+---------+---------------+---------------+
|       Aarav|Hyderabad| aarav@mail.com|[Dust, Peanuts]|
|       Priya|Bangalore| priya@mail.com|       [Pollen]|
|       Rahul|   Mumbai| rahul@mail.com|   [Dust, Milk]|
|       Meera|Hyderabad| meera@mail.com|      [Seafood]|
|      Vikram|   Mumbai|vikram@mail.com| [Pollen, Dust]|
+------------+---------+---------------+---------------+



In [129]:
# 10
profiles.join(patients,"patient_id").select("city",explode("allergies").alias("allergy")).groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|   Mumbai|    4|
|Hyderabad|    3|
+---------+-----+



## Bonus — Spark SQL Exercises

In [131]:
patients.createOrReplaceTempView('patients')
doctors.createOrReplaceTempView('doctors')
appointments.createOrReplaceTempView('appointments')
bills.createOrReplaceTempView('bills')
print('Temp views created')


Temp views created


In [132]:
# 1
spark.sql('SELECT * FROM patients').show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [133]:
# 2
spark.sql("SELECT * FROM patients WHERE city = 'Hyderabad'").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
+----------+------+---------+---+------+-----------------+



In [134]:
# 3
spark.sql('SELECT city, COUNT(*) as patient_count FROM patients GROUP BY city ORDER BY patient_count DESC').show()

+---------+-------------+
|     city|patient_count|
+---------+-------------+
|    Delhi|            3|
|Hyderabad|            3|
|Bangalore|            2|
|   Mumbai|            2|
|  Chennai|            1|
|     Pune|            1|
+---------+-------------+



In [135]:
# 4
spark.sql('SELECT specialization, COUNT(*) as doctor_count FROM doctors GROUP BY specialization').show()

+--------------+------------+
|specialization|doctor_count|
+--------------+------------+
|     Neurology|           1|
|   Dermatology|           2|
|    Cardiology|           2|
|    Pediatrics|           1|
|   Orthopedics|           2|
+--------------+------------+



In [136]:
# 5
spark.sql('''
    SELECT p.name, p.city, a.appointment_date, a.status
    FROM patients p
    JOIN appointments a ON p.patient_id = a.patient_id
''').show()

+------+---------+----------------+---------+
|  name|     city|appointment_date|   status|
+------+---------+----------------+---------+
| Aarav|Hyderabad|      2024-03-01|Completed|
| Priya|Bangalore|      2024-03-01|Completed|
| Rahul|   Mumbai|      2024-03-02|Completed|
| Sneha|    Delhi|      2024-03-02|  Pending|
| Kiran|  Chennai|      2024-03-03|Completed|
| Meera|Hyderabad|      2024-03-03|Completed|
|  Amit|     Pune|      2024-03-04|Cancelled|
|  Neha|    Delhi|      2024-03-04|Completed|
| Divya|Bangalore|      2024-03-05|Completed|
|Vikram|   Mumbai|      2024-03-05|Completed|
|Farhan|Hyderabad|      2024-03-06|  Pending|
|Simran|    Delhi|      2024-03-06|Completed|
| Aarav|Hyderabad|      2024-03-07|Completed|
| Rahul|   Mumbai|      2024-03-07|Completed|
| Meera|Hyderabad|      2024-03-08|Cancelled|
| Divya|Bangalore|      2024-03-08|Completed|
+------+---------+----------------+---------+



In [139]:
# 6
spark.sql('''
    SELECT d.doctor_name, d.specialization, a.appointment_date, a.status
    FROM doctors d
    JOIN appointments a ON d.doctor_id = a.doctor_id
''').show()

+-----------+--------------+----------------+---------+
|doctor_name|specialization|appointment_date|   status|
+-----------+--------------+----------------+---------+
|  Dr Sharma|    Cardiology|      2024-03-01|Completed|
|    Dr Iyer|   Dermatology|      2024-03-01|Completed|
|    Dr Khan|   Orthopedics|      2024-03-02|Completed|
|   Dr Reddy|    Pediatrics|      2024-03-02|  Pending|
|    Dr Nair|    Cardiology|      2024-03-03|Completed|
|   Dr Mehta|     Neurology|      2024-03-03|Completed|
|   Dr Verma|   Dermatology|      2024-03-04|Cancelled|
|     Dr Rao|   Orthopedics|      2024-03-04|Completed|
|    Dr Iyer|   Dermatology|      2024-03-05|Completed|
|    Dr Khan|   Orthopedics|      2024-03-05|Completed|
|  Dr Sharma|    Cardiology|      2024-03-06|  Pending|
|   Dr Reddy|    Pediatrics|      2024-03-06|Completed|
|   Dr Mehta|     Neurology|      2024-03-07|Completed|
|     Dr Rao|   Orthopedics|      2024-03-07|Completed|
|  Dr Sharma|    Cardiology|      2024-03-08|Can

In [140]:
# 7
spark.sql('''
    SELECT a.appointment_id, a.status, b.bill_amount, b.payment_status
    FROM appointments a
    JOIN bills b ON a.appointment_id = b.appointment_id
''').show()

+--------------+---------+-----------+--------------+
|appointment_id|   status|bill_amount|payment_status|
+--------------+---------+-----------+--------------+
|             1|Completed|       1200|          Paid|
|             2|Completed|        800|          Paid|
|             3|Completed|       1500|          Paid|
|             4|  Pending|        900|       Pending|
|             5|Completed|       1300|          Paid|
|             6|Completed|       2000|          Paid|
|             7|Cancelled|        850|     Cancelled|
|             8|Completed|       1400|          Paid|
|             9|Completed|        800|          Paid|
|            10|Completed|       1500|          Paid|
|            11|  Pending|       1200|       Pending|
|            12|Completed|        900|          Paid|
|            13|Completed|       2000|          Paid|
|            14|Completed|       1400|          Paid|
|            15|Cancelled|       1200|     Cancelled|
|            16|Completed|  

In [141]:
# 8
spark.sql('SELECT payment_mode, SUM(bill_amount) as total FROM bills GROUP BY payment_mode ORDER BY total DESC').show()

+------------+-----+
|payment_mode|total|
+------------+-----+
|         UPI| 6900|
| Credit Card| 6300|
|        Cash| 4450|
|  Debit Card| 2600|
+------------+-----+



In [142]:
# 9
spark.sql('''
    SELECT doctor_name, consultation_fee,
           RANK() OVER (ORDER BY consultation_fee DESC) as fee_rank
    FROM doctors
''').show()

+-----------+----------------+--------+
|doctor_name|consultation_fee|fee_rank|
+-----------+----------------+--------+
|   Dr Mehta|            2000|       1|
|    Dr Khan|            1500|       2|
|     Dr Rao|            1400|       3|
|    Dr Nair|            1300|       4|
|  Dr Sharma|            1200|       5|
|   Dr Reddy|             900|       6|
|   Dr Verma|             850|       7|
|    Dr Iyer|             800|       8|
+-----------+----------------+--------+



In [143]:
# 10
spark.sql('''
    SELECT p.name, SUM(b.bill_amount) as total_billed
    FROM patients p
    JOIN appointments a ON p.patient_id = a.patient_id
    JOIN bills b ON a.appointment_id = b.appointment_id
    GROUP BY p.name
    ORDER BY total_billed DESC
    LIMIT 1
''').show()

+-----+------------+
| name|total_billed|
+-----+------------+
|Meera|        3200|
+-----+------------+

